# Phase 2: Feature Engineering

**Goal:** build one pre-treatment feature table with one row per labelled customer.

The notebook needs the Parquet files from `01_experiment_audit.ipynb`. It runs in this order: check the SQL on known answers, confirm the transaction key, build both tables, validate them, and check them against the data dictionary.

`target` is never read here. `sql/02_feature_engineering.sql` reads `uplift_train.parquet` only through a view without that column.

Definitions are in `outputs/tables/data_dictionary.md`, and decisions are in `reports/decisions.md`.

## Setup

In [1]:
import os
import re
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)

TABLES = Path("outputs/tables")
PROCESSED = Path("data/processed")

PRIMARY_DATE = "2019-03-19 00:00:00"
SENSITIVITY_DATE = "2019-03-12 00:00:00"
PRIMARY_CUTOFF = f"TIMESTAMP '{PRIMARY_DATE}'"
SENSITIVITY_CUTOFF = f"TIMESTAMP '{SENSITIVITY_DATE}'"
EXPECTED_LABELLED_CUSTOMERS = 200_039
FEATURES_PRIMARY = PROCESSED / "customer_features.parquet"
FEATURES_SENSITIVITY = PROCESSED / "customer_features_sensitivity.parquet"

pd.set_option("display.max_columns", 40, "display.width", 200, "display.max_rows", 200, "display.max_colwidth", 70)

con = duckdb.connect()
con.execute(
    "SET memory_limit='5GB'; SET preserve_insertion_order=false; "
    "SET temp_directory='data/processed/.duckdb_tmp'"
)


def query(sql):
    return con.execute(sql).fetchdf()


def values_match(left, right):
    """True when two columns agree, treating nulls in the same position as equal."""
    return np.allclose(np.asarray(left, float), np.asarray(right, float), rtol=1e-9, atol=1e-6, equal_nan=True)

## 1. Known-answer check

The SQL runs on a small constructed dataset, and every feature must match a value worked out by hand. This includes a purchase exactly at the cutoff, which must be left out.

In [2]:
# Known answers on a small constructed dataset. It holds the cases most likely to break:
# a purchase exactly at the cutoff, a transaction_id shared by two clients, same-day
# card issues on either side of a purchase, a zero-sum transaction and a customer
# with no history. Customer D is unlabelled and must not appear.
fixture = duckdb.connect()
fixture.execute(Path("sql/02_feature_engineering.sql").read_text(encoding="utf-8"))

fixture_transactions = pd.DataFrame(
    [
        # client, transaction, datetime, sum, rows, regular received, regular spent, express received, express spent
        ("A", "T1", "2019-03-18 10:00:00", 100.0, 2, 5.0, -2.0, 1.0, -1.0),
        ("A", "T2", "2019-03-02 15:00:00", 50.0, 1, 3.0, 0.0, 0.0, 0.0),
        ("A", "T3", "2019-02-01 20:00:00", 30.0, 1, 0.0, 0.0, 0.0, 0.0),
        ("A", "POST", "2019-03-19 00:00:00", 999.0, 1, 9.0, -9.0, 9.0, -9.0),
        ("B", "T1", "2019-03-01 09:00:00", 70.0, 1, 0.0, 0.0, 0.0, 0.0),
        ("C", "POST2", "2019-03-20 12:00:00", 999.0, 1, 9.0, -9.0, 9.0, -9.0),
        ("E", "T5", "2019-03-05 09:00:00", 0.0, 1, 0.0, 0.0, 0.0, 0.0),
        ("D", "T6", "2019-03-05 09:00:00", 500.0, 1, 1.0, 0.0, 0.0, 0.0),
        # F sits on both sides of every window start. Doubling amounts make each sum unique.
        ("F", "F01", "2019-03-12 00:00:00", 1.0, 1, 0.0, 0.0, 0.0, 0.0),
        ("F", "F02", "2019-03-11 23:59:59", 2.0, 1, 0.0, 0.0, 0.0, 0.0),
        ("F", "F03", "2019-03-05 00:00:00", 4.0, 1, 0.0, 0.0, 0.0, 0.0),
        ("F", "F04", "2019-03-04 23:59:59", 8.0, 1, 0.0, 0.0, 0.0, 0.0),
        ("F", "F05", "2019-02-17 00:00:00", 16.0, 1, 0.0, -1.0, 0.0, -1.0),
        ("F", "F06", "2019-02-16 23:59:59", 32.0, 1, 0.0, -2.0, 0.0, -2.0),
        ("F", "F07", "2019-01-18 00:00:00", 64.0, 1, 0.0, 0.0, 0.0, 0.0),
        ("F", "F08", "2019-01-17 23:59:59", 128.0, 1, 0.0, 0.0, 0.0, 0.0),
        ("F", "F09", "2018-12-19 00:00:00", 256.0, 1, 0.0, 0.0, 0.0, 0.0),
        ("F", "F10", "2018-12-18 23:59:59", 512.0, 1, 0.0, 0.0, 0.0, 0.0),
        # G sits on both sides of every hour band and of the weekend.
        ("G", "G01", "2019-03-08 04:59:59", 1.0, 1, 0.0, 0.0, 0.0, 0.0),
        ("G", "G02", "2019-03-11 05:00:00", 1.0, 1, 0.0, 0.0, 0.0, 0.0),
        ("G", "G03", "2019-03-12 09:59:59", 1.0, 1, 0.0, 0.0, 0.0, 0.0),
        ("G", "G04", "2019-03-13 10:00:00", 1.0, 1, 0.0, 0.0, 0.0, 0.0),
        ("G", "G05", "2019-03-14 13:59:59", 1.0, 1, 0.0, 0.0, 0.0, 0.0),
        ("G", "G06", "2019-03-15 14:00:00", 1.0, 1, 0.0, 0.0, 0.0, 0.0),
        ("G", "G07", "2019-03-18 19:59:59", 1.0, 1, 0.0, 0.0, 0.0, 0.0),
        ("G", "G08", "2019-03-07 20:00:00", 1.0, 1, 0.0, 0.0, 0.0, 0.0),
        ("G", "G09", "2019-03-01 23:59:59", 1.0, 1, 0.0, 0.0, 0.0, 0.0),
        ("G", "G10", "2019-03-02 00:00:00", 1.0, 1, 0.0, 0.0, 0.0, 0.0),
        ("G", "G11", "2019-03-03 23:59:59", 1.0, 1, 0.0, 0.0, 0.0, 0.0),
        ("G", "G12", "2019-03-04 00:00:00", 1.0, 1, 0.0, 0.0, 0.0, 0.0),
    ],
    columns=[
        "client_id", "transaction_id", "transaction_datetime", "purchase_sum", "n_rows",
        "regular_points_received", "regular_points_spent", "express_points_received", "express_points_spent",
    ],
).assign(transaction_datetime=lambda frame: pd.to_datetime(frame["transaction_datetime"]))
fixture_purchases = pd.DataFrame(
    [
        ("A", "T1", "2019-03-18 10:00:00", "s1", "p1"),
        ("A", "T1", "2019-03-18 10:00:00", "s1", "p2"),
        ("A", "T2", "2019-03-02 15:00:00", "s2", "p1"),
        ("A", "T3", "2019-02-01 20:00:00", "s1", "p3"),
        ("A", "POST", "2019-03-19 00:00:00", "s9", "p9"),
        ("B", "T1", "2019-03-01 09:00:00", "s1", "p4"),
        ("C", "POST2", "2019-03-20 12:00:00", "s9", "p9"),
        ("E", "T5", "2019-03-05 09:00:00", "s1", "p1"),
        ("D", "T6", "2019-03-05 09:00:00", "s1", "p1"),
        ("F", "F01", "2019-03-12 00:00:00", "s1", "p1"),
        ("G", "G01", "2019-03-08 04:59:59", "s1", "p1"),
    ],
    columns=["client_id", "transaction_id", "transaction_datetime", "store_id", "product_id"],
).assign(transaction_datetime=lambda frame: pd.to_datetime(frame["transaction_datetime"]))
fixture_clients = pd.DataFrame(
    [
        ("A", "2018-03-19 00:00:00", 30, "F"),
        ("B", "2019-03-01 12:00:00", 200, "U"),
        ("C", "2019-03-20 00:00:00", 14, "M"),
        ("E", "2019-03-05 08:00:00", 100, "F"),
        ("D", "2018-01-01 00:00:00", 40, "F"),
        ("F", "2018-01-01 00:00:00", 50, "M"),
        ("G", "2018-01-01 00:00:00", 50, "M"),
    ],
    columns=["client_id", "first_issue_date", "age", "gender"],
).assign(first_issue_date=lambda frame: pd.to_datetime(frame["first_issue_date"]))
fixture_groups = pd.DataFrame({"client_id": ["A", "B", "C", "E", "F", "G"]})

for view, frame in (
    ("transactions", fixture_transactions),
    ("purchases", fixture_purchases),
    ("clients", fixture_clients),
    ("customer_groups", fixture_groups),
):
    fixture.register(f"{view}_fixture", frame)
    fixture.execute(f"CREATE OR REPLACE VIEW {view} AS SELECT * FROM {view}_fixture")

known = fixture.execute("SELECT * FROM customer_features(TIMESTAMP '2019-03-19 00:00:00')").fetchdf().set_index("client_id")
assert list(known.index) == ["A", "B", "C", "E", "F", "G"], "Population or ordering is wrong"

N = np.nan
expected_features = {
    #                                     A                                B              C    E
    "days_since_last_purchase":           [0,                              17,            N,   13],
    "days_since_first_observed_purchase": [45,                             17,            N,   13],
    "txn_count_hist":                     [3,                              1,             0,   1],
    "txn_count_7d":                       [1,                              0,             0,   0],
    "txn_count_14d":                      [1,                              0,             0,   1],
    "txn_count_30d":                      [2,                              1,             0,   1],
    "txn_count_60d":                      [3,                              1,             0,   1],
    "txn_count_90d":                      [3,                              1,             0,   1],
    "active_days_hist":                   [3,                              1,             0,   1],
    "avg_gap_days":                       [22.5,                           N,             N,   N],
    "spend_hist":                         [180,                            70,            0,   0],
    "spend_30d":                          [150,                            70,            0,   0],
    "spend_60d":                          [180,                            70,            0,   0],
    "spend_90d":                          [180,                            70,            0,   0],
    "avg_basket_value":                   [60,                             70,            N,   0],
    "max_basket_value":                   [100,                            70,            N,   0],
    "std_basket_value":                   [np.sqrt(1300),                  N,             N,   N],
    "txn_trend_30d":                      [np.log(3 / 2),                  np.log(2),     0,   np.log(2)],
    "spend_trend_30d":                    [np.log(151 / 31),               np.log(71),    0,   0],
    "avg_lines_per_basket":               [4 / 3,                          1,             N,   1],
    "distinct_products_hist":             [3,                              1,             0,   1],
    "distinct_stores_hist":               [2,                              1,             0,   1],
    "weekend_txn_share":                  [1 / 3,                          0,             N,   0],
    "txn_share_05_09":                    [0,                              1,             N,   1],
    "txn_share_10_13":                    [1 / 3,                          0,             N,   0],
    "txn_share_14_19":                    [1 / 3,                          0,             N,   0],
    "regular_points_received_hist":       [8,                              0,             0,   0],
    "regular_points_spent_hist":          [2,                              0,             0,   0],
    "express_points_received_hist":       [1,                              0,             0,   0],
    "express_points_spent_hist":          [1,                              0,             0,   0],
    "regular_points_spent_30d":           [2,                              0,             0,   0],
    "express_points_spent_30d":           [1,                              0,             0,   0],
    "age":                                [30,                             N,             14,  100],
    "age_invalid_flg":                    [0,                              1,             0,   0],
    "gender_f":                           [1,                              0,             0,   1],
    "gender_m":                           [0,                              0,             1,   0],
    "gender_u":                           [0,                              1,             0,   0],
    "days_since_first_issue":             [365,                            17,            N,   13],
    "pre_issue_purchase_flg":             [0,                              1,             0,   0],
}
assert list(known.columns) == list(expected_features), "Feature columns or their order changed"
for feature, values in expected_features.items():
    actual = known.loc[["A", "B", "C", "E"], feature].astype(float).to_numpy()
    assert np.allclose(actual, np.array(values, float), rtol=1e-9, equal_nan=True), f"{feature}: {actual} expected {values}"
# Spent points are shown as positive, including a true positive zero.
assert not np.signbit(known["regular_points_spent_hist"].astype(float).to_numpy()).any()

# Window starts are inclusive: a purchase at exactly cutoff - N days counts, and one second earlier does not.
amounts = 2.0 ** np.arange(10)
expected_boundaries = {
    "days_since_last_purchase": 7,
    "days_since_first_observed_purchase": 90,
    "txn_count_hist": 10,
    "txn_count_7d": 1,
    "txn_count_14d": 3,
    "txn_count_30d": 5,
    "txn_count_60d": 7,
    "txn_count_90d": 9,
    "active_days_hist": 10,
    "avg_gap_days": 84 / 9,
    "spend_hist": 1023,
    "spend_30d": 31,
    "spend_60d": 127,
    "spend_90d": 511,
    "avg_basket_value": 102.3,
    "max_basket_value": 512,
    "std_basket_value": amounts.std(ddof=1),
    "txn_trend_30d": np.log(6 / 3),
    "spend_trend_30d": np.log(32 / 97),
    "weekend_txn_share": 2 / 10,
    "regular_points_spent_hist": 3,
    "regular_points_spent_30d": 1,
    "express_points_spent_hist": 3,
    "express_points_spent_30d": 1,
    "days_since_first_issue": 442,
}
for feature, value in expected_boundaries.items():
    assert np.isclose(float(known.loc["F", feature]), value, rtol=1e-9), f"F {feature}: {known.loc['F', feature]} expected {value}"

# Hour bands include whole hours, so 09:59:59 is in 05-09 and 20:00:00 is in the remainder.
# Saturday 00:00:00 and Sunday 23:59:59 are weekend; Friday 23:59:59 and Monday 00:00:00 are not.
expected_timing = {"txn_share_05_09": 2 / 12, "txn_share_10_13": 2 / 12, "txn_share_14_19": 2 / 12, "weekend_txn_share": 2 / 12}
for feature, value in expected_timing.items():
    assert np.isclose(float(known.loc["G", feature]), value, rtol=1e-9), f"G {feature}: {known.loc['G', feature]} expected {value}"
fixture.close()
print(f"Known-answer check passed for {len(expected_features)} features on {len(known)} customers.")

Known-answer check passed for 39 features on 6 customers.


## 2. Transaction key and transaction-level fields

`purchase_sum` and the four points fields repeat across a transaction's product rows. Features use one value per `(client_id, transaction_id)`, so that key must be right and those fields must be constant within it.

In [3]:
con.execute(Path("sql/02_feature_engineering.sql").read_text(encoding="utf-8"))

key_check = query(
    """
    SELECT
        count(*) AS transaction_rows,
        count(DISTINCT (client_id, transaction_id)) AS distinct_pairs,
        count(DISTINCT transaction_id) AS distinct_transaction_ids
    FROM transactions
    """
).iloc[0]
assert key_check["transaction_rows"] == key_check["distinct_pairs"] == 8_045_229
assert key_check["distinct_transaction_ids"] == 8_045_201, "transaction_id alone is not the key: 28 values are shared"

# Measured on the product-level rows, independently of transactions.parquet.
not_constant = query(
    """
    SELECT count(*) AS n
    FROM (
        SELECT client_id, transaction_id
        FROM purchases
        GROUP BY client_id, transaction_id
        HAVING min(purchase_sum) <> max(purchase_sum)
            OR min(regular_points_received) <> max(regular_points_received)
            OR min(regular_points_spent) <> max(regular_points_spent)
            OR min(express_points_received) <> max(express_points_received)
            OR min(express_points_spent) <> max(express_points_spent)
    )
    """
)["n"].iloc[0]
assert not_constant == 0, "A transaction-level field varies within a transaction"

print(key_check.to_string())
print("Transactions where purchase_sum or a points field varies:", int(not_constant))

transaction_rows            8045229
distinct_pairs              8045229
distinct_transaction_ids    8045201
Transactions where purchase_sum or a points field varies: 0


## 3. Build both tables

In [4]:
for path, cutoff in ((FEATURES_PRIMARY, PRIMARY_CUTOFF), (FEATURES_SENSITIVITY, SENSITIVITY_CUTOFF)):
    con.execute(
        f"COPY (SELECT * FROM customer_features({cutoff})) TO '{path.as_posix()}' (FORMAT parquet, COMPRESSION zstd)"
    )

features = query(f"SELECT * FROM '{FEATURES_PRIMARY.as_posix()}' ORDER BY client_id").set_index("client_id")
sensitivity = query(f"SELECT * FROM '{FEATURES_SENSITIVITY.as_posix()}' ORDER BY client_id").set_index("client_id")

FEATURE_NAMES = list(features.columns)
assert len(FEATURE_NAMES) == 39
assert list(sensitivity.columns) == FEATURE_NAMES
assert features.index.equals(sensitivity.index)

for table in (features, sensitivity):
    assert len(table) == EXPECTED_LABELLED_CUSTOMERS
    assert table.index.is_unique
    assert not {"target", "treatment_flg", "split", "fold", "first_redeem_date"} & set(table.columns)

print(f"{len(FEATURE_NAMES)} features for {len(features):,} customers in each table")

39 features for 200,039 customers in each table


## 4. Validation

### Age rule

The rule must match Phase 1 exactly: 14 to 100 inclusive is valid, and anything else is null with a flag.

In [5]:
baseline = query(
    """
    SELECT client_id, age, age_invalid_flg, gender, days_since_first_issue, pre_issue_purchase_flg,
           transaction_count, total_spend, days_since_last_purchase
    FROM 'data/processed/baseline_customers.parquet'
    ORDER BY client_id
    """
).set_index("client_id")
assert baseline.index.equals(features.index)

assert values_match(features["age"], baseline["age"])
assert (features["age_invalid_flg"] == baseline["age_invalid_flg"]).all()
assert features["age"].dropna().between(14, 100).all()
assert features["age"].isna().equals(features["age_invalid_flg"] == 1)
assert features["age_invalid_flg"].sum() == 883

raw_invalid = query(
    """
    SELECT count(*) AS n
    FROM clients JOIN customer_groups USING (client_id)
    WHERE age NOT BETWEEN 14 AND 100
    """
)["n"].iloc[0]
assert raw_invalid == 883
print("Age rule matches Phase 1. Invalid ages among labelled customers:", int(raw_invalid))

Age rule matches Phase 1. Invalid ages among labelled customers: 883


### Reconciliation with Phase 1 and the raw rows

The primary table must match `baseline_customers.parquet`. Both tables must also match customer-level values recalculated from the product-level rows, keeping one value per `(client_id, transaction_id)`. Double counting, a wrong key or a purchase at or after the cutoff would fail here.

In [6]:
for column, baseline_column in {
    "txn_count_hist": "transaction_count",
    "spend_hist": "total_spend",
    "days_since_last_purchase": "days_since_last_purchase",
    "days_since_first_issue": "days_since_first_issue",
    "pre_issue_purchase_flg": "pre_issue_purchase_flg",
}.items():
    assert values_match(features[column], baseline[baseline_column]), f"{column} differs from baseline_customers"
for gender, column in (("F", "gender_f"), ("M", "gender_m"), ("U", "gender_u")):
    assert ((baseline["gender"] == gender).astype(int) == features[column]).all()


def recalculated_from_rows(cutoff):
    return query(
        f"""
        WITH one_value AS (
            SELECT
                client_id,
                transaction_id,
                any_value(transaction_datetime) AS transaction_datetime,
                any_value(purchase_sum) AS purchase_sum,
                any_value(regular_points_received) AS regular_points_received,
                any_value(regular_points_spent) AS regular_points_spent,
                any_value(express_points_received) AS express_points_received,
                any_value(express_points_spent) AS express_points_spent
            FROM purchases
            WHERE transaction_datetime < {cutoff}
            GROUP BY client_id, transaction_id
        )
        SELECT
            g.client_id,
            count(t.client_id) AS txn_count_hist,
            coalesce(sum(t.purchase_sum), 0) AS spend_hist,
            date_sub('day', max(t.transaction_datetime), {cutoff}) AS days_since_last_purchase,
            coalesce(sum(t.regular_points_received), 0) AS regular_points_received_hist,
            coalesce(-sum(t.regular_points_spent), 0) AS regular_points_spent_hist,
            coalesce(sum(t.express_points_received), 0) AS express_points_received_hist,
            coalesce(-sum(t.express_points_spent), 0) AS express_points_spent_hist
        FROM customer_groups AS g
        LEFT JOIN one_value AS t USING (client_id)
        GROUP BY g.client_id
        ORDER BY g.client_id
        """
    ).set_index("client_id")


for table, cutoff in ((features, PRIMARY_CUTOFF), (sensitivity, SENSITIVITY_CUTOFF)):
    recalculated = recalculated_from_rows(cutoff)
    assert recalculated.index.equals(table.index)
    for column in recalculated.columns:
        assert values_match(table[column], recalculated[column]), f"{column} differs from the raw rows at {cutoff}"

print("Both tables match the raw rows, and the primary table matches baseline_customers.")

Both tables match the raw rows, and the primary table matches baseline_customers.


### No purchase at or after the cutoff

The checks above and the known-answer test cover this directly. These add a check across the two tables: the primary table's history minus its last seven days must equal the sensitivity table's history for every customer.

In [7]:
assert (features["days_since_last_purchase"].dropna() >= 0).all()
assert (sensitivity["days_since_last_purchase"].dropna() >= 0).all()
assert (features["txn_count_hist"] - features["txn_count_7d"] == sensitivity["txn_count_hist"]).all()

latest_used = query(
    f"""
    SELECT
        max(transaction_datetime) FILTER (WHERE transaction_datetime < {PRIMARY_CUTOFF}) AS latest_primary,
        max(transaction_datetime) FILTER (WHERE transaction_datetime < {SENSITIVITY_CUTOFF}) AS latest_sensitivity,
        count(*) FILTER (WHERE transaction_datetime >= {PRIMARY_CUTOFF}) AS rows_at_or_after_primary
    FROM purchases
    """
).iloc[0]
assert latest_used["rows_at_or_after_primary"] == 0
latest_used

latest_primary              2019-03-18 23:40:03
latest_sensitivity          2019-03-11 23:51:31
rows_at_or_after_primary                      0
Name: 0, dtype: object

### Value checks

In [8]:
COUNT_WINDOWS = ["txn_count_7d", "txn_count_14d", "txn_count_30d", "txn_count_60d", "txn_count_90d", "txn_count_hist"]
SPEND_WINDOWS = ["spend_30d", "spend_60d", "spend_90d", "spend_hist"]
SHARES = ["weekend_txn_share", "txn_share_05_09", "txn_share_10_13", "txn_share_14_19"]
POINTS = [column for column in FEATURE_NAMES if "points" in column]

for table in (features, sensitivity):
    assert not np.isinf(table.astype(float).to_numpy()).any()
    # Longer windows can never hold less than shorter ones.
    for shorter, longer in zip(COUNT_WINDOWS, COUNT_WINDOWS[1:]):
        assert (table[shorter] <= table[longer]).all(), f"{shorter} exceeds {longer}"
    for shorter, longer in zip(SPEND_WINDOWS, SPEND_WINDOWS[1:]):
        assert (table[shorter] <= table[longer] + 1e-6).all(), f"{shorter} exceeds {longer}"
    assert (table[POINTS] >= 0).all().all(), "Points must not be negative"
    assert not np.signbit(table[POINTS].astype(float).to_numpy()).any(), "Points hold a negative zero"
    for share in SHARES:
        assert table[share].dropna().between(0, 1).all(), f"{share} is outside 0 to 1"
    assert (table[["txn_share_05_09", "txn_share_10_13", "txn_share_14_19"]].sum(axis=1) <= 1 + 1e-9).all()
    assert (table[["gender_f", "gender_m", "gender_u"]].sum(axis=1) == 1).all()
    assert table["avg_gap_days"].isna().equals(table["active_days_hist"] < 2)
    assert table["std_basket_value"].isna().equals(table["txn_count_hist"] < 2)
    assert (table["active_days_hist"] <= table["txn_count_hist"]).all()

# At the primary cutoff, everyone has a purchase on or after 2019-02-24.
assert features["days_since_last_purchase"].notna().all()
assert features["days_since_last_purchase"].max() == 22
assert sensitivity["days_since_last_purchase"].max() == 109
assert features[["txn_count_30d", "txn_count_60d", "txn_count_90d"]].min().min() >= 1
assert features[["spend_30d", "spend_60d", "spend_90d"]].min().min() > 0
assert (sensitivity["distinct_products_hist"] <= features["distinct_products_hist"]).all()
assert (sensitivity["distinct_stores_hist"] <= features["distinct_stores_hist"]).all()
print("Value checks passed.")

Value checks passed.


### `pre_issue_purchase_flg`

The flag compares timestamps. A same-day first purchase before the issue time is 1, and one at or after the issue time is 0. This splits the flagged customers by whether they bought on the issue date itself.

In [9]:
def pre_issue_split(cutoff):
    return query(
        f"""
        WITH first_purchase AS (
            SELECT client_id, min(transaction_datetime) AS first_transaction
            FROM transactions
            WHERE transaction_datetime < {cutoff}
            GROUP BY client_id
        )
        SELECT
            sum((first_transaction < first_issue_date)::INT) AS flagged,
            sum((first_transaction < first_issue_date AND first_transaction::DATE = first_issue_date::DATE)::INT)
                AS flagged_same_day,
            sum((first_transaction::DATE < first_issue_date::DATE)::INT) AS flagged_earlier_date,
            sum((first_transaction >= first_issue_date AND first_transaction::DATE = first_issue_date::DATE)::INT)
                AS same_day_at_or_after_issue
        FROM customer_groups
        JOIN clients USING (client_id)
        LEFT JOIN first_purchase USING (client_id)
        """
    ).iloc[0].astype(int)


pre_issue = pd.DataFrame({"primary": pre_issue_split(PRIMARY_CUTOFF), "sensitivity": pre_issue_split(SENSITIVITY_CUTOFF)})
assert pre_issue.loc["flagged", "primary"] == features["pre_issue_purchase_flg"].sum()
assert pre_issue.loc["flagged", "sensitivity"] == sensitivity["pre_issue_purchase_flg"].sum()
assert (pre_issue.loc["flagged_same_day"] + pre_issue.loc["flagged_earlier_date"] == pre_issue.loc["flagged"]).all()

# The data dictionary and reports/decisions.md quote these values.
assert pre_issue["primary"].tolist() == [23_532, 23_515, 17, 5]
pre_issue

,primary,sensitivity
flagged,23532,23378
flagged_same_day,23515,23361
flagged_earlier_date,17,17
same_day_at_or_after_issue,5,5


### Sensitivity table pattern

The 753 customers with no purchase before 2019-03-12 must have zeros and nulls where the data dictionary says. `days_since_first_issue` must be exactly seven days lower than in the primary table, and null only for cards issued from 2019-03-12 to 2019-03-18. Age and gender must not change.

In [10]:
no_history = sensitivity["txn_count_hist"] == 0
assert no_history.sum() == 753
assert sensitivity["days_since_last_purchase"].isna().equals(no_history)

ZERO_WHEN_NO_HISTORY = (
    COUNT_WINDOWS + SPEND_WINDOWS + POINTS
    + ["active_days_hist", "distinct_products_hist", "distinct_stores_hist", "txn_trend_30d", "spend_trend_30d", "pre_issue_purchase_flg"]
)
NULL_WHEN_NO_HISTORY = SHARES + [
    "days_since_last_purchase", "days_since_first_observed_purchase", "avg_gap_days",
    "avg_basket_value", "max_basket_value", "std_basket_value", "avg_lines_per_basket",
]
CLIENT_FIELDS = ["age", "age_invalid_flg", "gender_f", "gender_m", "gender_u", "days_since_first_issue"]
assert set(ZERO_WHEN_NO_HISTORY + NULL_WHEN_NO_HISTORY + CLIENT_FIELDS) == set(FEATURE_NAMES)
assert len(ZERO_WHEN_NO_HISTORY + NULL_WHEN_NO_HISTORY + CLIENT_FIELDS) == len(FEATURE_NAMES)

assert (sensitivity.loc[no_history, ZERO_WHEN_NO_HISTORY] == 0).all().all()
assert sensitivity.loc[no_history, NULL_WHEN_NO_HISTORY].isna().all().all()

# Customers with history have every basket and timing value, so nulls come only from the pattern above.
assert sensitivity.loc[~no_history, SHARES + ["avg_basket_value", "max_basket_value", "avg_lines_per_basket"]].notna().all().all()

for column in ("age", "age_invalid_flg", "gender_f", "gender_m", "gender_u"):
    assert values_match(sensitivity[column], features[column]), f"{column} changed between cutoffs"

issued = query(
    f"""
    SELECT client_id, first_issue_date >= {SENSITIVITY_CUTOFF} AND first_issue_date < {PRIMARY_CUTOFF} AS issued_in_final_week
    FROM customer_groups JOIN clients USING (client_id)
    ORDER BY client_id
    """
).set_index("client_id")["issued_in_final_week"]
assert issued.index.equals(features.index)
assert issued.sum() == 154
assert features["days_since_first_issue"].notna().all()
assert sensitivity["days_since_first_issue"].isna().equals(issued)
shift = (features["days_since_first_issue"] - sensitivity["days_since_first_issue"]).dropna()
assert (shift == 7).all()
assert (issued & ~no_history).sum() == 0, "The log says all final-week cards are among the customers with no history"

print("Customers with no purchase before the sensitivity cutoff:", int(no_history.sum()))
print("Cards issued from 2019-03-12 to 2019-03-18:", int(issued.sum()))
print("Customers in both groups:", int((no_history & issued).sum()))

Customers with no purchase before the sensitivity cutoff: 753
Cards issued from 2019-03-12 to 2019-03-18: 154
Customers in both groups: 154


### Edge cases recalculated in pandas

Each feature is worked out again from the product-level rows with separate pandas code, for one customer from each case most likely to break. The customers are chosen by lowest `client_id`, not by looking at their results.

In [11]:
def lowest(mask):
    return features.index[mask][0]


# Every case below must hold before the sensitivity cutoff, so it is exercised at both cutoffs.
def first_purchase_client(condition):
    return query(
        f"""
        SELECT client_id
        FROM transactions JOIN clients USING (client_id) JOIN customer_groups USING (client_id)
        WHERE transaction_datetime < {SENSITIVITY_CUTOFF}
        GROUP BY client_id, first_issue_date
        HAVING {condition}
        ORDER BY client_id
        LIMIT 1
        """
    )["client_id"].iloc[0]


earlier_date_client = first_purchase_client("min(transaction_datetime)::DATE < first_issue_date::DATE")
same_day_client = first_purchase_client(
    "min(transaction_datetime) < first_issue_date AND min(transaction_datetime)::DATE = first_issue_date::DATE"
)
zero_sum_client = query(
    f"""
    SELECT min(client_id) AS client_id
    FROM transactions JOIN customer_groups USING (client_id)
    WHERE purchase_sum = 0 AND transaction_datetime < {SENSITIVITY_CUTOFF}
    """
)["client_id"].iloc[0]
shared_id_client = query(
    f"""
    SELECT min(client_id) AS client_id
    FROM transactions JOIN customer_groups USING (client_id)
    WHERE transaction_datetime < {SENSITIVITY_CUTOFF}
      AND transaction_id IN (
          SELECT transaction_id FROM transactions GROUP BY transaction_id HAVING count(*) > 1
      )
    """
)["client_id"].iloc[0]
for client in (earlier_date_client, same_day_client):
    assert features.loc[client, "pre_issue_purchase_flg"] == 1 and sensitivity.loc[client, "pre_issue_purchase_flg"] == 1

EDGE_CASES = {
    "One transaction": lowest(features["txn_count_hist"] == 1),
    "Invalid age": lowest(features["age_invalid_flg"] == 1),
    "Purchase on an earlier date than issue": earlier_date_client,
    "Purchase before issue on the same day": same_day_client,
    "Zero-sum transaction": zero_sum_client,
    "Shared transaction_id": shared_id_client,
    "No history at the sensitivity cutoff": lowest(no_history),
    "Card issued in the final week": lowest(issued),
}

id_list = ", ".join(f"'{client}'" for client in EDGE_CASES.values())
edge_rows = query(f"SELECT * FROM purchases WHERE client_id IN ({id_list})")
edge_clients = query(f"SELECT * FROM clients WHERE client_id IN ({id_list})").set_index("client_id")


def recalculate_by_hand(client_id, cutoff_date):
    cutoff = pd.Timestamp(cutoff_date)
    rows = edge_rows[(edge_rows["client_id"] == client_id) & (edge_rows["transaction_datetime"] < cutoff)]
    client = edge_clients.loc[client_id]
    baskets = rows.groupby("transaction_id").agg(
        when=("transaction_datetime", "first"),
        amount=("purchase_sum", "first"),
        regular_received=("regular_points_received", "first"),
        regular_spent=("regular_points_spent", "first"),
        express_received=("express_points_received", "first"),
        express_spent=("express_points_spent", "first"),
        lines=("product_id", "size"),
    )

    def days_before_cutoff(stamp):
        return (cutoff - stamp).days

    def last(days):
        return baskets[baskets["when"] >= cutoff - pd.Timedelta(days=days)]

    previous = baskets[
        (baskets["when"] >= cutoff - pd.Timedelta(days=60)) & (baskets["when"] < cutoff - pd.Timedelta(days=30))
    ]
    has_history = len(baskets) > 0
    dates = baskets["when"].dt.normalize()
    active_days = dates.nunique()
    hours = baskets["when"].dt.hour
    valid_age = 14 <= client["age"] <= 100

    return {
        "days_since_last_purchase": days_before_cutoff(baskets["when"].max()) if has_history else np.nan,
        "days_since_first_observed_purchase": days_before_cutoff(baskets["when"].min()) if has_history else np.nan,
        "txn_count_hist": len(baskets),
        "txn_count_7d": len(last(7)),
        "txn_count_14d": len(last(14)),
        "txn_count_30d": len(last(30)),
        "txn_count_60d": len(last(60)),
        "txn_count_90d": len(last(90)),
        "active_days_hist": active_days,
        "avg_gap_days": (dates.max() - dates.min()).days / (active_days - 1) if active_days >= 2 else np.nan,
        "spend_hist": baskets["amount"].sum(),
        "spend_30d": last(30)["amount"].sum(),
        "spend_60d": last(60)["amount"].sum(),
        "spend_90d": last(90)["amount"].sum(),
        "avg_basket_value": baskets["amount"].mean(),
        "max_basket_value": baskets["amount"].max(),
        "std_basket_value": baskets["amount"].std(ddof=1) if len(baskets) >= 2 else np.nan,
        "txn_trend_30d": np.log((len(last(30)) + 1) / (len(previous) + 1)),
        "spend_trend_30d": np.log((last(30)["amount"].sum() + 1) / (previous["amount"].sum() + 1)),
        "avg_lines_per_basket": baskets["lines"].mean(),
        "distinct_products_hist": rows["product_id"].nunique(),
        "distinct_stores_hist": rows["store_id"].nunique(),
        "weekend_txn_share": (baskets["when"].dt.day_of_week >= 5).mean() if has_history else np.nan,
        "txn_share_05_09": hours.between(5, 9).mean() if has_history else np.nan,
        "txn_share_10_13": hours.between(10, 13).mean() if has_history else np.nan,
        "txn_share_14_19": hours.between(14, 19).mean() if has_history else np.nan,
        "regular_points_received_hist": baskets["regular_received"].sum(),
        "regular_points_spent_hist": -baskets["regular_spent"].sum(),
        "express_points_received_hist": baskets["express_received"].sum(),
        "express_points_spent_hist": -baskets["express_spent"].sum(),
        "regular_points_spent_30d": -last(30)["regular_spent"].sum(),
        "express_points_spent_30d": -last(30)["express_spent"].sum(),
        "age": client["age"] if valid_age else np.nan,
        "age_invalid_flg": int(not valid_age),
        "gender_f": int(client["gender"] == "F"),
        "gender_m": int(client["gender"] == "M"),
        "gender_u": int(client["gender"] == "U"),
        "days_since_first_issue": days_before_cutoff(client["first_issue_date"]) if client["first_issue_date"] < cutoff else np.nan,
        "pre_issue_purchase_flg": int(has_history and baskets["when"].min() < client["first_issue_date"]),
    }


rows = []
for case, client_id in EDGE_CASES.items():
    for name, cutoff_date, table in (("Primary", PRIMARY_DATE, features), ("Sensitivity", SENSITIVITY_DATE, sensitivity)):
        by_hand = recalculate_by_hand(client_id, cutoff_date)
        assert list(by_hand) == FEATURE_NAMES
        stored = table.loc[client_id, FEATURE_NAMES].astype(float).to_numpy()
        calculated = np.array(list(by_hand.values()), float)
        differs = [f for f, a, b in zip(FEATURE_NAMES, stored, calculated) if not np.isclose(a, b, rtol=1e-9, atol=1e-6, equal_nan=True)]
        assert not differs, f"{case} at the {name} cutoff differs on {differs}"
        rows.append({"case": case, "client_id": client_id, "cutoff": name, "txn_count_hist": by_hand["txn_count_hist"], "features_matching": len(FEATURE_NAMES)})

edge_case_check = pd.DataFrame(rows)
assert edge_case_check["client_id"].nunique() >= 5
edge_case_check.to_csv(TABLES / "feature_edge_cases.csv", index=False)
edge_case_check

,case,client_id,cutoff,txn_count_hist,features_matching
0,One transaction,0009e6bafa,Primary,1,39
1,One transaction,0009e6bafa,Sensitivity,1,39
2,Invalid age,000220a0a7,Primary,15,39
3,Invalid age,000220a0a7,Sensitivity,14,39
4,Purchase on an earlier date than issue,020cb514d8,Primary,51,39
5,Purchase on an earlier date than issue,020cb514d8,Sensitivity,48,39
6,Purchase before issue on the same day,00035a21d9,Primary,3,39
7,Purchase before issue on the same day,00035a21d9,Sensitivity,3,39
8,Zero-sum transaction,011fef655c,Primary,19,39
9,Zero-sum transaction,011fef655c,Sensitivity,18,39


## 5. Data dictionary and profile

The dictionary must list exactly the stored columns, in order, and carry the agreed label on the three express features.

In [12]:
dictionary_text = (TABLES / "data_dictionary.md").read_text(encoding="utf-8")
dictionary_rows = re.findall(r"^\| `(\w+)` \| \w+ \|.*\| ([^|]+) \|$", dictionary_text, re.M)
assert [name for name, _ in dictionary_rows] == FEATURE_NAMES, "The data dictionary does not match the table"

EXPRESS_FEATURES = {"express_points_received_hist", "express_points_spent_hist", "express_points_spent_30d"}
assert {name for name, status in dictionary_rows if status == "Cleared in Phase 1; inclusion set by the treatment-check rule"} == EXPRESS_FEATURES
assert {status for name, status in dictionary_rows if name not in EXPRESS_FEATURES} == {"Core"}
assert "`transaction_datetime < 2019-03-19 00:00`" in dictionary_text
assert "`transaction_datetime < 2019-03-12 00:00`" in dictionary_text


def profile(table, cutoff_name):
    numeric = table.astype(float)
    return pd.DataFrame(
        {
            "cutoff": cutoff_name,
            "feature": FEATURE_NAMES,
            "null_count": numeric.isna().sum().to_numpy(),
            "null_share": numeric.isna().mean().to_numpy(),
            "minimum": numeric.min().to_numpy(),
            "median": numeric.median().to_numpy(),
            "mean": numeric.mean().to_numpy(),
            "maximum": numeric.max().to_numpy(),
        }
    )


feature_profile = pd.concat([profile(features, "Primary"), profile(sensitivity, "Sensitivity")], ignore_index=True)
feature_profile.to_csv(TABLES / "feature_profile.csv", index=False)
feature_profile[feature_profile["cutoff"] == "Primary"].drop(columns="cutoff").round(3)

,feature,null_count,null_share,minimum,median,mean,maximum
0,days_since_last_purchase,0,0.000,0.000,4.000,5.829,22.000
1,days_since_first_observed_purchase,0,0.000,3.000,110.000,97.269,117.000
2,txn_count_hist,0,0.000,1.000,15.000,20.121,320.000
3,txn_count_7d,0,0.000,0.000,1.000,1.376,44.000
4,txn_count_14d,0,0.000,0.000,2.000,2.898,69.000
5,txn_count_30d,0,0.000,1.000,4.000,6.115,119.000
6,txn_count_60d,0,0.000,1.000,9.000,11.340,223.000
7,txn_count_90d,0,0.000,1.000,12.000,15.989,282.000
8,active_days_hist,0,0.000,1.000,14.000,17.874,115.000
9,avg_gap_days,4085,0.020,1.000,6.765,10.057,113.000


### Measurements quoted in the documents

The data dictionary and `reports/decisions.md` quote these values, so a change must stop the run.

In [13]:
hour_counts = query(
    """
    SELECT hour(transaction_datetime) AS hour, count(*) AS n
    FROM transactions SEMI JOIN customer_groups USING (client_id)
    GROUP BY 1
    """
).set_index("hour")["n"].astype(float)
hour_shares = 100 * hour_counts / hour_counts.sum()
late_evening = hour_shares[hour_shares.index >= 20].sum()
outside_bands = hour_shares[(hour_shares.index >= 20) | (hour_shares.index < 5)].sum()

quantity = query(
    """
    SELECT
        100 * avg((product_quantity = 0)::INT) AS zero_quantity_rows,
        100 * avg((trn_sum_from_iss > 0)::INT) FILTER (WHERE product_quantity = 0) AS zero_quantity_with_amount
    FROM purchases SEMI JOIN customer_groups USING (client_id)
    """
).iloc[0]
level_1_spend = query(
    """
    SELECT level_1, sum(trn_sum_from_iss) AS amount
    FROM purchases SEMI JOIN customer_groups USING (client_id)
    JOIN 'data/processed/products.parquet' USING (product_id)
    GROUP BY level_1
    ORDER BY amount DESC
    """
)
top_two_level_1 = 100 * level_1_spend["amount"].iloc[:2].sum() / level_1_spend["amount"].sum()
first_purchase_early = 100 * (features["days_since_first_observed_purchase"] >= 110).mean()

assert set(hour_shares.nlargest(2).index) == {9, 14}
assert round(late_evening, 2) == 0.16
assert round(outside_bands, 1) == 0.6
assert round(first_purchase_early, 1) == 50.4
assert round(quantity["zero_quantity_rows"], 1) == 5.9
assert round(quantity["zero_quantity_with_amount"], 1) == 99.7
assert round(top_two_level_1) == 98

print(f"Busiest recorded hours: {sorted(hour_shares.nlargest(2).index)}")
print(f"Transactions from 20:00 to 23:59: {late_evening:.2f}%")
print(f"Transactions from 20:00 to 04:59: {outside_bands:.1f}%")
print(f"Customers whose first recorded purchase is in about the first eight days: {first_purchase_early:.1f}%")
print(f"Product rows with zero quantity: {quantity['zero_quantity_rows']:.1f}%, of which {quantity['zero_quantity_with_amount']:.1f}% have a positive amount")
print(f"Spend share of the two largest level_1 values: {top_two_level_1:.1f}%")

Busiest recorded hours: [9, 14]
Transactions from 20:00 to 23:59: 0.16%
Transactions from 20:00 to 04:59: 0.6%
Customers whose first recorded purchase is in about the first eight days: 50.4%
Product rows with zero quantity: 5.9%, of which 99.7% have a positive amount
Spend share of the two largest level_1 values: 98.1%


## Phase 2 status so far

- **Tables:** `customer_features.parquet` and `customer_features_sensitivity.parquet`, each with 39 features for all 200,039 customers. They hold no treatment, outcome or split columns.
- **Transaction key:** `(client_id, transaction_id)` is confirmed, and `purchase_sum` and all four points fields are constant within every transaction.
- **Age:** the rule matches Phase 1, with 883 invalid.
- **`pre_issue_purchase_flg`:** compares timestamps. Of 23,532 flagged customers, 23,515 bought on the issue date itself.
- **Not yet run:** the balance rerun and the treatment-prediction check. Their rules in `reports/decisions.md` must be committed first.